**Question Bank**

**1. Why MTL instead of many STLs?**

Three reasons it does better:
- Allows us to understand common structures rather than indepedent rediscovering disease association many times indepedently, easy to flag using one set of parameters to show which diseases might be of high risk for a person.
- Allows for shared SHAP values which can show us which features have universal risk factors versus disease specifc. This is instead of many independent SHAP vectors.
- Helps with the sparse signal problem, aka when there are some rare diseases, it might have little to train from but having other diseases there allows for richer gradient updates.


One reason where it doesn't do as well:
- MTL may do wrose than STL from a pure prediction aspect on diseases with lots of data.



**2. Why use MTL-NN instead of Multi Output RF?**

- RF requires K independent models and cannot share representations across tasks. The NN shared trunk jointly optimises all K predictions, letting diseases borrow signal from each other, which is particularly useful for rare diseases.
- We need the masking to avoid the model taking shortcuts, this can be cleanly done in he MTL design where head k sees all disease flags except its own while RF has no equivalent mechanism.
- SHAP on a jointly trained trunk reveals which features are universally important versus disease specific. K separate RF models just give K independent rankings, which is a weaker and less novel finding.

**3. Why is the SHAP on the jointly trained trunk so cool?**

It looks at which features drive each persons prediction, which means you could also look at:
- Identify which patients are high risk for multiple diseases simultaneously (high SHAP across all columns)
- Find subgroups where certain features matter more or less (e.g. BMI matters more for older patients)
- Flag individual patients where the model is uncertain or driven by unusual features

**Summary**
- We get a feature importance score for every input feature for every disease simultaneously, not just one disease at a time. This produces a features x diseases matrix that reveals which demographics are universal risk factors versus disease specific ones.
- Because the trunk is jointly optimised across all K diseases, the SHAP values reflect a richer representation than running K separate models. The trunk has learned what is globally useful, so SHAP on it measures something more meaningful than single task importance.
- I believe it is directly interpretable for policy. You can show a clinician or policymaker that age and BMI drive risk across all conditions while smoking is specific to respiratory disease, using numbers not intuition.
- Ialso believe this is a novel contribution. No existing paper applies SHAP to a jointly trained shared trunk on survey data, which is part of what makes your project distinct from the literature

**4. What is the beenfit of doing MTL on cross-sectional survey data?**

- Time series is more predictive but cross-sectional is publicly available. EHR data requires hospital access, ethics approval, and data sharing agreements which most researchers never get.
- Cross-sectional surveys like NHANES are nationally representative by design. EHR data only captures people who went to hospital, which may induce selection bias.
- Cross-sectional results are more easily reproducible compaer to EHR studies which are very diffiuclt to.
- Most importantly it is clinically appropriate, for example A GP screening tool works from a single visit not 10 years of records, so cross-sectional is actually the right format for deployment as a useable tool.

**5. Why are we using binary disease flags as explicit input features to predict other diseases?**

**The benefits are:**
- Captures comorbidity structure directly since diseases co-occur in patterns and this lets the model exploit that signal explicitly rather than hoping demographics alone encode it.
- Clinically realistic as at a GP visit you often know some diagnoses already, so using them as inputs mirrors how a doctor actually reasons.
- It offers a cheap signal; binary flags are free to include, require no extra data collection, and are already in the survey.
- Improves prediction of rare diseases since a rare condition with weak demographic signal can borrow strength from its comorbidities.

**However, there are some costs:**
- We have temporal ambiguity as cross-sectional data gives no ordering, so you cannot tell if disease A preceded B or vice versa we are just modelling association not causation.
- There is circular prediction risk because if two diseases share diagnostic criteria or one is definitionally linked to another, including it as a feature is close to cheating.
- Test time availability. In a real screening setting the whole point is you do not know the diagnoses yet, so using them as inputs undermines the use case.
- Label leakage as disease A may be a proxy for a latent factor that also causes disease B, inflating apparent predictive performance without genuine signal.
- Class imbalance amplification as a rare disease flag that is mostly zero adds sparse, noisy signal that can confuse the model if not handled carefully.

****

****

****

****